In [4]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/Student_Depression_Dataset.csv")
df.head()

,Gender,Age,Family_background,Relationship_status,Marital_status,Living_style,Current_semester,CGPA,friend_circle,Health_condition,Cost_without_study,Home_division,Depression
0,Female,22,Middle class,Single,Unmarried,Bachelor,8,3.43,Yes,Critical,10113,Chittagong,Yes
1,Male,19,Middle class,Single,Unmarried,Bachelor,2,3.70,Yes,Abnormal,13815,Chittagong,Yes
2,Male,23,Middle class,Single,Unmarried,Family,8,3.28,No,Abnormal,15450,Dhaka,Yes
3,Male,20,Middle class,Complicated,Unmarried,Family,2,3.77,No,Good,17413,Dhaka,No
4,Female,21,Middle class,Single,Unmarried,Family,1,4.00,No,Abnormal,16693,Dhaka,Yes


In [5]:
categorial_cols = df.drop(columns = ['Depression']).select_dtypes(include = 'str').columns.to_list()
numerical_cols = df.select_dtypes(include = ['int', 'float']).columns.to_list()
categorial_cols, numerical_cols

(['Gender',
  'Family_background',
  'Relationship_status',
  'Marital_status',
  'Living_style',
  'friend_circle',
  'Health_condition',
  'Home_division'],
 ['Age', 'Current_semester', 'CGPA', 'Cost_without_study'])

In [41]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer


preprocessor = ColumnTransformer(transformers=[
    ('encoding', OneHotEncoder(sparse_output= False, dtype=int, handle_unknown='ignore', drop = 'first'), categorial_cols),
    ('scaling', StandardScaler(), numerical_cols)
], remainder='passthrough', verbose_feature_names_out=False)

X = df.drop(columns="Depression")

X = pd.DataFrame(
    preprocessor.fit_transform(X),
    columns=preprocessor.get_feature_names_out()
)

le = LabelEncoder()
y = le.fit_transform(df["Depression"])

In [45]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

In [46]:
X_train.shape,y_train.shape

((3124, 23), (3124,))

# Tuning the Optimizer

In [2]:
from keras.models import Sequential
from keras.layers import Input, Dense, Dropout

In [14]:
def build_model(hp):
    model = Sequential([
        Input(shape = (23,)),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(8, activation='relu'),
        Dense(1, activation='sigmoid'),
    ])

    optimizer = hp.Choice('optimizer', values = ['adam', 'sgd', 'rmsprop', 'adadelta'])

    model.compile(optimizer = optimizer, loss = "binary_crossentropy", metrics = ['accuracy'])

    return model

In [55]:
import kerastuner as kt

tuner = kt.RandomSearch(
    build_model,
    objective= 'val_accuracy',
    max_trials=10,
    project_name='optimizer_tuning_v2_20261029',
    overwrite=True
)

tuner.search(X_train, y_train, epochs = 5, validation_data = (X_test, y_test))

Reloading Tuner from .\untitled_project\tuner0.json


In [56]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'sgd'}

In [ ]:
tuner.results_summary()

In [59]:
best_model = tuner.get_best_models(num_models=1)[0]

best_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         3,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │           264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,681 (53.44 KB)

 Trainable params: 13,681 (53.44 KB)

 Non-trainable params: 0 (0.00 B)

# 2. How Many neurons in one hidden Layer.

In [64]:
def build_model(hp):
    units = hp.Int('units', min_value = 32,max_value =  512, step = 32)
    model = Sequential([
        Input(shape = (23,)),
        Dense(units=units, activation='relu'),
        Dense(1, activation='sigmoid'),
    ])

    model.compile(optimizer = 'sgd', loss = "binary_crossentropy", metrics = ['accuracy'])

    return model

tuner2 = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=20,
    project_name='neuron_count_tuning_v2_20261029',
    overwrite=True
)

tuner2.search(X_train, y_train, epochs = 10, validation_data = (X_test, y_test))

Trial 16 Complete [00h 00m 12s]
val_accuracy: 0.8850746154785156

Best val_accuracy So Far: 0.9089552164077759
Total elapsed time: 00h 03m 18s


In [65]:
tuner2.get_best_hyperparameters()[0].values

{'units': 288}

In [67]:
best_model = tuner2.get_best_models()[0]

best_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 288)            │         6,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           289 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,201 (28.13 KB)

 Trainable params: 7,201 (28.13 KB)

 Non-trainable params: 0 (0.00 B)

# 3. Tuning the Number of Hidden Layers

In [68]:
def build_layer_count_model(hp):
    num_layers = hp.Int('num_layers', min_value=1, max_value=5)
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))

    for _ in range(num_layers):
        model.add(Dense(64, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

layer_tuner = kt.RandomSearch(
    build_layer_count_model,
    objective='val_accuracy',
    max_trials=5,
    project_name='hidden_layer_count_tuning_v1_20261029',
    overwrite=True
)

layer_tuner.search(
    X_train, y_train, epochs=10,
    validation_data=(X_test, y_test)
)

Trial 5 Complete [00h 00m 14s]
val_accuracy: 0.9768656492233276

Best val_accuracy So Far: 0.9880596995353699
Total elapsed time: 00h 01m 20s


In [69]:
layer_tuner.get_best_hyperparameters()[0].values

{'num_layers': 5}

In [70]:
layer_tuner.results_summary()
best_layer_model = layer_tuner.get_best_models(num_models=1)[0]
best_layer_model.summary()

Results summary
Results in .\hidden_layer_count_tuning_v1_20261029
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 1 summary
Hyperparameters:
num_layers: 5
Score: 0.9880596995353699

Trial 0 summary
Hyperparameters:
num_layers: 4
Score: 0.987313449382782

Trial 2 summary
Hyperparameters:
num_layers: 3
Score: 0.987313449382782

Trial 3 summary
Hyperparameters:
num_layers: 2
Score: 0.9850746393203735

Trial 4 summary
Hyperparameters:
num_layers: 1
Score: 0.9768656492233276


c:\Users\Keval\OneDrive\Desktop\Deep-Learning-Journey\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 26 variables. 
  saveable.load_own_variables(store)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,241 (71.25 KB)

 Trainable params: 18,241 (71.25 KB)

 Non-trainable params: 0 (0.00 B)

# 4. Tuning the Full Model

In [ ]:
def build_combined_model(hp):
    num_layers = hp.Int('num_layers', min_value=1, max_value=5)
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))

    for i in range(num_layers):
        units = hp.Int(f'units_{i + 1}', min_value=32, max_value=512, step=32)
        model.add(Dense(units=units, activation='relu'))
        if hp.Boolean("dropout"):
            model.add(Dropout(rate=0.25))

    optimizer = hp.Choice(
        'optimizer', values=['adam', 'sgd', 'rmsprop', 'adadelta']
    )
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

combined_tuner = kt.RandomSearch(
    build_combined_model,
    objective='val_accuracy',
    max_trials=30,
    project_name='combined_model_tuning_v1_20261029',
    overwrite=True
)

combined_tuner.search(
    X_train, y_train, epochs=10,
    validation_data=(X_test, y_test)
)

In [ ]:
combined_tuner.get_best_hyperparameters()[0].values

In [ ]:
combined_tuner.results_summary()
best_combined_model = combined_tuner.get_best_models(num_models=1)[0]
best_combined_model.summary()

# 5. Tuning the batch size

Override fit to tune batch size and epochs through a HyperModel class:

In [ ]:
class MyHyperModel(kt.HyperModel):
    def build(self, hp):
        return build_model(hp)

    def fit(self, hp, model, *args, **kwargs):
        return model.fit(
            *args,
            batch_size=hp.Choice("batch_size", [16, 32, 64, 128]),
            **kwargs,
        )

tuner = kt.RandomSearch(MyHyperModel(), objective="val_loss", max_trials=20)